# Bloc 1 — Qu'est-ce qu'un modèle de décision ? (10 min)

**Situation.** Le support client d'un opérateur télécom (fictif) reçoit tous ses messages dans une seule boîte. Un conseiller expérimenté range *« on m'a prélevé deux fois 29,99 € ce mois-ci »* en *facturation* en une demi-seconde. L'objectif : reproduire ce tri **localement**, sans envoyer les messages des clients à un tiers, et savoir quand le modèle hésite.

**Question au groupe, avant de lancer quoi que ce soit** : le conseiller a-t-il *rédigé* une réponse, ou *choisi* dans une liste qu'il connaît par cœur ?

In [ ]:
# Préparation (rien à modifier) — exécutez avec Maj+Entrée
import sys; sys.path.insert(0, "..")   # le paquet atelier/ (installé par installer.sh ; ceci au cas où)
import matplotlib.pyplot as plt
import pandas as pd
import wyat
import atelier as at                  # même boîte à outils que la commande `atelier` et les exemples/
pd.set_option("display.max_colwidth", 90)
modele = at.charger_modele()

## 1. Une décision, avec ses probabilités

In [ ]:
message = "Bonjour, on m'a prélevé deux fois 29,99 € ce mois-ci, vous pouvez me rembourser ?"
d = at.decider(modele, message)
print("Choix du modèle :", d.selected)
print("Marge entre le 1er et le 2e choix :", round(d.probability_margin, 3))
at.probas(d)

In [ ]:
def graphique(decision, titre=""):
    p = at.probas(decision).sort_values()
    p.plot.barh(color=["#2a9d8f" if k == decision.selected else "#bbbbbb" for k in p.index], figsize=(6, 2.6))
    plt.xlim(0, 1); plt.title(titre[:80]); plt.xlabel("probabilité"); plt.show()

graphique(d, message)

## 2. Plusieurs messages : certains sont faciles, d'autres non
Regardez la **marge** : proche de 1, le modèle est catégorique ; proche de 0, il hésite entre deux options.

In [ ]:
messages = [
    "la box clignote rouge depuis hier soir",
    "je veux arrêter mon abonnement svp",
    "Mon colis est marqué livré mais je n'ai rien reçu",
    "J'ai résilié en août et vous continuez de me prélever !!!",   # deux sujets à la fois
    "bonjour",                                                    # aucun sujet
]
lignes = []
for m in messages:
    d = at.decider(modele, m)
    lignes.append({"message": m, "choix": d.selected, "proba": round(d.probabilities_by_id[d.selected], 3),
                   "2e choix": d.ranking[1], "marge": round(d.probability_margin, 3)})
pd.DataFrame(lignes)

### Ce que vous venez de voir
**Question** : dans ce tableau, quels messages le modèle traite-t-il sans hésiter ? Qu'ont-ils en commun dans la colonne *marge* ? Et le message « J'ai résilié… vous continuez de me prélever » ?

*Réponse attendue* : les messages à un seul sujet ont une marge proche de 1 ; le message à deux sujets (résiliation **et** prélèvement) a une marge de 0,46 : il hésite entre deux services, comme un humain.

**Maintenant, les mots.** Ce réflexe porte un nom : le **Système 1** (Kahneman), rapide et sans raisonnement déroulé, par opposition au **Système 2**, lent et étape par étape.
- Un grand modèle de langage (ChatGPT, Claude, Mistral…) **écrit** une réponse mot à mot : plutôt Système 2, lent et coûteux.
- Un **modèle de décision** comme *wyat-1-800M* **choisit** : on lui donne un message et une **liste fermée d'options**, il note chaque option et rend **une probabilité par option**. Environ 0,15 s par décision sur un Mac Apple Silicon (≈ 1 s sur un PC Linux sans GPU), en local.
- La **marge** = probabilité du 1er choix − probabilité du 2e. Proche de 0 : il hésite.

> Le modèle **choisit** une option ; il n'**exécute** rien. C'est votre code qui décide de l'action.

## 3. L'ordre des options ne change rien
Chaque option est lue dans une branche séparée : inverser la liste donne les mêmes probabilités.

In [ ]:
inverse = dict(reversed(list(at.OPTIONS.items())))
m = "la box clignote rouge depuis hier soir"
a = at.decider(modele, m, at.OPTIONS).probabilities_by_id
b = at.decider(modele, m, inverse).probabilities_by_id
pd.DataFrame({"ordre normal": a, "ordre inversé": b}).round(4)

## 4. Teaser : la formulation des options compte énormément
On ajoute une option **attrape-tout** « pas sûr : demander à une personne ». Elle a l'air prudente… et pourtant elle aspire des messages clairs. Le modèle ne lit **que** ce que vous lui écrivez : le libellé d'une option *est* sa définition.

In [ ]:
avec_attrape_tout = dict(at.OPTIONS, humain="Pas sûr ou pas clair : transmettre à une personne")
for m in ["la box clignote rouge depuis hier soir", "Ma nouvelle carte SIM n'est pas activée, ça fait 48h", "comment on résilie ?"]:
    sans = at.decider(modele, m, at.OPTIONS)
    avec = at.decider(modele, m, avec_attrape_tout)
    print(f"{m[:50]:<52} sans attrape-tout → {sans.selected:<12} | avec → {avec.selected:<12} (p={avec.probabilities_by_id[avec.selected]:.2f})")

**Question** : la box qui clignote rouge, c'est évident pour un conseiller. Pourquoi le modèle l'envoie-t-il soudain à « humain » ? *Réponse attendue* : l'option « pas sûr » ressemble à tout message un peu flou ; il ne connaît **que** les mots écrits dans les options. **Le libellé d'une option est sa définition.**

**À retenir**
- Une décision = une question, des options fermées, une probabilité par option.
- Ces probabilités ne sont **pas** une confiance garantie : on les mesurera (bloc 5).
- « Passer à un humain » se fait mieux avec une **règle sur la marge** qu'avec une option attrape-tout : c'est ce qu'on construira.

### Bonus : noter sur une échelle (Score)

In [ ]:
msg = "Plus d'internet depuis ce matin et je télétravaille, c'est urgent !!"
urgence = modele.decide(wyat.Request.score("Quelle est l'urgence de ce message ?",
                                           ["Peut attendre plusieurs jours.", "À traiter aujourd'hui.", "À traiter immédiatement."], context=msg))
print("Score (0 = peut attendre … 2 = immédiat) : valeur attendue =", round(urgence.expected_ordinal_value, 2))
print(at.probas(urgence))

Le troisième type, **Noul** (oui / non), est au bloc 6 : c'est aussi le plus sensible à la formulation.

---
**Étape 1, point de contrôle** (`missions/1_premiere_decision/MISSION.md`) : le même objectif, à atteindre comme vous voulez (ce notebook, la commande `atelier`, ou un script).
Vérification sur dev : `atelier verifier 1` dans un terminal, ou la cellule ci-dessous.

In [ ]:
from pathlib import Path
from atelier.missions import preparer, verifier
preparer(Path("../travail"))   # crée travail/ avec les fichiers de départ s'ils manquent (n'écrase rien)
print(verifier(1, globals().get("modele"), Path("../travail")).texte())   # réutilise le modèle déjà chargé